# Generation-input exploratory data analysis

This notebook inspects canonical pre-COMSOL input cases.  
Prepare or extend inputs through a Dataset plan run, then rerun the notebook. Completed solver outputs can be inspected through `EDA_VIEW = "generated_source"` in [Source / Dataset EDA](02_eda_datasets.ipynb).



## Field and support reference

The atlas uses admitted persisted arrays and deterministic diagnostics only. Full-resolution summaries precede display sampling. Boundary/scalar/schedule support is not converted into a domain map.

| Quantity / stored key | Meaning and unit | Evidence and display support |
|---|---|---|
| `x`, `y` | Physical coordinates [m] | Persisted grid; geometric support. |
| `Kxx`, `Kxy`, `Kyy` | Symmetric permeability tensor [m²] | Persisted domain arrays; positive logarithmic diagonal scales and signed cross-component scale. |
| `K_min`, `K_max`, `K_anisotropy` | Principal permeabilities [m²] and their ratio [1] | Deterministic tensor diagnostics on the admitted grid. |
| Principal direction | Major eigenvector axis [degree modulo180] | Derived only where anisotropy defines a unique axis; fixed cyclic scale. An isotropic cell makes this optional map unavailable. |
| `kappa` | Generator's lognormal scalar permeability [m²] | Producer computes it before tensor construction but does not persist its original array separately. The tensor's geometric-mean principal permeability is derivable; it must not be described as a separately stored backbone. |
| `z`, `z_bg` | Combined/transformed structure and unperturbed latent background [dimensionless] | Exact stochastic arrays are not persisted in this contract; unavailable, never regenerated for display. |
| `eps_bed` (porosity) | Bed void fraction [1] | Persisted domain array. `eps_bed_ref` is a scalar calibration reference, not the spatial field. |
| `rho_bu_dry` | Dry bulk density [kg/m³ of bed] | Exactly derived from the admitted reference density/porosity relation. Spatial variation can be mapped; constant density is metadata. |
| `rho_gr_dry` | Intrinsic dry grain density [kg/m³ of grain] | No separate persisted 2D input under this name in the inspected contract. Do not relabel bulk density as intrinsic density. |
| `X_0_db_field` / diagnostic X0 dry basis | Initial water per dry grain mass [kg/kg] | Persisted domain field. Wet-basis moisture is a different nonlinear quantity. |
| `w_gr0` | Initial volumetric granular water [kg/m³ of bed] | Derived as dry bulk density × dry-basis initial moisture; not a surface/internal compartment. |
| `phi_eq` | Equilibrium relative humidity [1] | Derived through the admitted Oswin parameters and initial temperature; not an independent random humidity field. |
| `p_in_bc` | Inlet pressure [Pa] | Real inlet profile only; excluded from atlas domain maps despite the producer's grid-shaped input representation. |
| `T_in_bc`, `omega_in_bc`, `phi_in_bc` | Inlet temperature [K; °C display], humidity ratio [kg/kg], relative humidity [1] | Persisted temperature/humidity-ratio schedule nodes; relative humidity is a canonical psychrometric derivation. Preserve regular/startup supports and schedule semantics. |
| Material scalars, masks and geometry | Contract-declared units / categorical support | Scalars remain tables; the inspected input contract has a rectangular grid and no independent persisted phase-mask array to invent. |

Published Dataset representations differ: steady permeability inputs are dimensionless encoded ratios, while transient state/conditioning and delta targets use physical units. Inspect those actual model-facing tensors in [Dataset EDA](02_eda_datasets.ipynb); training-fitted normalization is a separate contract.


## Prepare input cases

A Dataset plan run prepares the canonical inputs of its Source before any solver starts ([Datasets](../docs/04_datasets.md)). Rerun this notebook after the run has prepared or extended them.

See [Generation](../docs/03_generation.md#input-only-generation-and-status) for input-only preparation of a campaign YAML, input selection, regime filtering, and all-cases generation.

In [ ]:
from pathlib import Path

from IPython.display import display as show

from src import common
from src.analysis import generation_inputs

# None discovers canonical Source inputs under the configured storage root.
# Set an explicit storage root only to inspect a separate scientific storage tree.
INPUT_EDA_STORAGE_ROOT: Path | None = None

STORAGE_ROOT = (
    common.paths.get_storage_root()
    if INPUT_EDA_STORAGE_ROOT is None
    else INPUT_EDA_STORAGE_ROOT
)

print(f"Generation-input EDA source: {STORAGE_ROOT}")

In [ ]:
DISPLAY_SPATIAL_STRIDE = "auto"  # 1, 2, 4, or auto; maps only, full-grid statistics

workspace = generation_inputs.workspace.prepare_generation_input_eda_workspace(
    storage_root=STORAGE_ROOT,
    display_spatial_stride=DISPLAY_SPATIAL_STRIDE,
    title="Generation-input EDA",
)
print(workspace.summary_text)

In [ ]:
if workspace.panel is not None:
    show(workspace.panel)

## Failures by parameter

Every terminal campaign candidate at its sampled value, one panel per parameter and one row per material. Successful candidates are grey; failed ones sit just above their material's row, coloured by the typed failure reason (for example `stalled`, `timed_out`, `solver_error`). The same rows are in each campaign's `cases.csv`. The picture shows recorded association only, not a causal effect.

In [ ]:
if workspace.outcomes.rows:
    outcome_table = generation_inputs.sources.campaign_outcome_table(workspace.outcomes.rows)
    show(
        generation_inputs.plots.overview.failures_by_parameter(
            outcome_table, generation_inputs.sources.campaign_outcome_parameters(outcome_table)
        )
    )